DuckDB 제대로 쓰기

In [2]:
import duckdb
import pandas as pd

In [3]:
# DB 연결
con = duckdb.connect(database='D:/Assets/BinanceFuturesData/binancefuturesdata.duckdb', read_only=True)

In [6]:
# DB 연결 해제
con.close()

In [17]:
# 타임스탬프(ms) -> 날짜
pd.to_datetime(1744801200000, unit='ms')

Timestamp('2025-04-16 11:00:00')

In [5]:
r1 = con.execute("SELECT * FROM symbol where name = 'BTCSTUSDT'").df()
r1

,name,liquidation_fee,listing_date,max_price,min_price,tick_size,max_quantity,min_quantity,step_size,price_precision,quantity_precision,underlying_type
0,BTCSTUSDT,0.04,1614754800000,100000.0,0.668,0.001,1000000.0,0.1,0.1,3,1,Coin


In [11]:
r2 = con.execute("select * from quote where symbol = 'BTCSTUSDT' order by timestamp desc limit 100").df()
r2

,symbol,interval,timestamp,open,high,low,close,volume,quote_volume,taker_buy_volume,taker_buy_quote_volume,trade_count
0,BTCSTUSDT,1,1718590260000,319.408,319.408,319.408,319.408,0.0,0.0,0.0,0.0,0
1,BTCSTUSDT,1,1718590200000,319.408,319.408,319.408,319.408,0.0,0.0,0.0,0.0,0
2,BTCSTUSDT,1,1718590140000,319.408,319.408,319.408,319.408,0.0,0.0,0.0,0.0,0
3,BTCSTUSDT,1,1718590080000,319.408,319.408,319.408,319.408,0.0,0.0,0.0,0.0,0
4,BTCSTUSDT,1,1718590020000,319.408,319.408,319.408,319.408,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
95,BTCSTUSDT,1,1718584560000,319.408,319.408,319.408,319.408,0.0,0.0,0.0,0.0,0
96,BTCSTUSDT,1,1718584500000,319.408,319.408,319.408,319.408,0.0,0.0,0.0,0.0,0
97,BTCSTUSDT,1,1718584440000,319.408,319.408,319.408,319.408,0.0,0.0,0.0,0.0,0
98,BTCSTUSDT,1,1718584380000,319.408,319.408,319.408,319.408,0.0,0.0,0.0,0.0,0


In [5]:
r3 = con.execute("PRAGMA table_info(quote)").df()
r3

,cid,name,type,notnull,dflt_value,pk
0,0,symbol,VARCHAR,True,None,True
1,1,interval,TINYINT,True,None,True
2,2,timestamp,BIGINT,True,None,True
3,3,open,"DECIMAL(18,8)",False,None,False
4,4,high,"DECIMAL(18,8)",False,None,False
5,5,low,"DECIMAL(18,8)",False,None,False
6,6,close,"DECIMAL(18,8)",False,None,False
7,7,volume,"DECIMAL(24,8)",False,None,False
8,8,quote_volume,"DECIMAL(18,8)",False,None,False
9,9,taker_buy_volume,"DECIMAL(24,8)",False,None,False


In [ ]:
#1. 전체 행 수와 NULL 개수 한눈에 보기
result = con.sql("""
                 SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(open)   AS null_open,
    COUNT(*) - COUNT(high)   AS null_high,
    COUNT(*) - COUNT(low)    AS null_low,
    COUNT(*) - COUNT(close)  AS null_close,
    COUNT(*) - COUNT(volume) AS null_volume,
    COUNT(*) - COUNT(timestamp) AS null_timestamp  -- 이건 거의 0이어야 함
FROM quote;   -- ← 실제 테이블명으로 변경
                 """).df()
result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,null_open,null_high,null_low,null_close,null_volume,null_timestamp
0,651762541,0,0,0,0,0,0


In [6]:
#2. symbol + interval 별로 데이터가 얼마나 깨졌는지 (가장 유용!)
result = con.sql("""
                 SELECT 
    symbol,
    COUNT(*) AS row_count,
    COUNT(*) FILTER(WHERE open IS NULL OR close IS NULL) AS null_ohlc_count,
    MIN(timestamp) AS first_ts,
    MAX(timestamp) AS last_ts,
    (MAX(timestamp) - MIN(timestamp)) / 1000 / 60 AS total_minutes_span  -- 분 단위
FROM quote
GROUP BY symbol
ORDER BY null_ohlc_count DESC, row_count DESC;
                 """).df()
result

,symbol,row_count,null_ohlc_count,first_ts,last_ts,total_minutes_span
0,BTCUSDT,3330237,0,1567969800000,1767783960000,3330236.0
1,ETHUSDT,3215722,0,1574840700000,1767783960000,3215721.0
2,BCHUSDT,3183970,0,1576745820000,1767783960000,3183969.0
3,XRPUSDT,3158086,0,1578298860000,1767783960000,3158085.0
4,LTCUSDT,3153778,0,1578557340000,1767783960000,3153777.0
...,...,...,...,...,...,...
607,IRUSDT,24525,0,1766313000000,1767784440000,24524.0
608,BREVUSDT,11550,0,1767091500000,1767784440000,11549.0
609,COLLECTUSDT,9960,0,1767186900000,1767784440000,9959.0
610,MAGMAUSDT,9945,0,1767187800000,1767784440000,9944.0


In [ ]:
# 변동률 임계값 설정 (예: 0.05 = 5%)
threshold = 1.0

# 1.0 으로 조회시 나온 심볼
# AERGOUSDT
# BULLAUSDT
# CVXUSDT
# CYBERUSDT
# FORMUSDT
# INUSDT
# LITUSDT
# PLUMEUSDT

query = f"""
WITH price_diffs AS (
    SELECT 
        symbol,
        timestamp,
        close,
        LAG(close) OVER (PARTITION BY symbol ORDER BY timestamp) as prev_close
    FROM quote
)
SELECT 
    symbol,
    timestamp,
    prev_close,
    close,
    ABS(close - prev_close) / prev_close as change_rate
FROM price_diffs
WHERE ABS(close - prev_close) / prev_close > {threshold}
ORDER BY symbol, timestamp;
"""

spikes_df = con.execute(query).df()
spikes_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,symbol,timestamp,prev_close,close,change_rate
0,AERGOUSDT,1744801200000,0.06790,0.35258,4.192636
1,BULLAUSDT,1760131440000,0.02298,0.04904,1.134030
2,CVXUSDT,1668366600000,3.87300,8.76500,1.263104
3,CVXUSDT,1753270200000,2.37400,4.97000,1.093513
4,CYBERUSDT,1754983800000,1.88000,4.47300,1.379255
5,FORMUSDT,1760131620000,0.21230,0.44220,1.082902
6,INUSDT,1760076000000,0.12252,0.25949,1.117940
7,LITUSDT,1766511000000,0.59200,3.87900,5.552365
8,PLUMEUSDT,1760131380000,0.01510,0.03245,1.149007
